In [19]:
import torch
from torch import nn

In [2]:
x = torch.randn(2, 5, 64)

In [3]:
B,T,d_model = x.shape

In [8]:
def split_heads(x,num_heads):
  B,T,d_model = x.shape
  d_head = d_model // num_heads
  x = (x.view(B,T,num_heads,-1)).transpose(1,2)
  return x.contiguous()

In [16]:
def merge_heads(x):
  B,num_heads,T,d_head = x.shape
  x = x.transpose(1,2).reshape(B,T,-1)
  return x

In [17]:
print(split_heads(x,4).shape)

torch.Size([2, 4, 5, 16])


In [18]:
heads = split_heads(x,4)
print(merge_heads(heads).shape)

torch.Size([2, 5, 64])


In [20]:
def masked_attention(q, k, v, mask=None):
    dk = (q.size(-1)) ** 0.5
    scores = (q @ k.transpose(-2, -1)) / dk
    if mask is not None:
        scores = scores.masked_fill(~mask, float('-inf'))
    weights = torch.softmax(scores, dim=-1)
    output = weights @ v
    return output, weights

In [21]:
class MultiHeadAttention(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.d_model = d_model
    self.num_heads = num_heads

    self.q_proj = nn.Linear(d_model,d_model)
    self.k_proj = nn.Linear(d_model,d_model)
    self.v_proj = nn.Linear(d_model,d_model)
    self.out_proj = nn.Linear(d_model,d_model)

  def forward(self,x,mask=None):
    q = self.q_proj(x)
    k = self.k_proj(x)
    v = self.v_proj(x)

    q = split_heads(q,self.num_heads)
    k = split_heads(k,self.num_heads)
    v = split_heads(v,self.num_heads)

    if mask is not None:
      if mask.dim() == 3:
        mask = mask.unsqueeze(1)
    attn_output, attn_weights = masked_attention(q, k, v, mask)
    attn_output = merge_heads(attn_output)
    return self.out_proj(attn_output)

In [22]:
class SimpleTransformationBlock(nn.Module):
  def __init__(self,d_model,num_heads):
    super().__init__()
    self.attn = MultiHeadAttention(d_model,num_heads)
    self.norm = nn.LayerNorm(d_model)

  def forward(self,x,mask=None):
    x_norm = nn.LayerNorm(x)
    attn_out = self.attn(x_norm,mask)
    out = x + attn_out
    return out